# MART_P02_TOP_ESPECIALIDADES

**P02. Quais são as 5 especialidades mais demandadas na 1ª (Maceió) e na 2ª (Arapiraca)
Macrorregião de Saúde?** *(dashboard)*

- Demanda = internações dos **residentes** de cada macrorregião, pela especialidade do leito.
- Ranking com `ROW_NUMBER()` por macrorregião; `PCT_MACRORREGIAO` = participação da
  especialidade no total da macrorregião.

**Estratégia de carga:** full (`TRUNCATE` + `INSERT /*+ APPEND */`), idempotente.

In [ ]:
import sys
sys.path.append("..")
from dw_utils import recarregar, consultar

In [ ]:
SQL = """
INSERT /*+ APPEND */ INTO MART_P02_TOP_ESPECIALIDADES
    (MACRORREGIAO, NR_RANKING, NOME_ESPECIALIDADE, QTD_INTERNACOES, PCT_MACRORREGIAO)
WITH base AS (
    SELECT m.MACRORREGIAO, e.NOME_ESPECIALIDADE, SUM(f.QTD_INTERNACAO) AS QTD
    FROM FATO_INTERNACAO f
    JOIN DIM_MUNICIPIO m     ON m.SK_MUNICIPIO = f.SK_MUNICIPIO_RESIDENCIA
    JOIN DIM_ESPECIALIDADE e ON e.SK_ESPECIALIDADE = f.SK_ESPECIALIDADE
    WHERE m.UF = 'AL' AND m.MACRORREGIAO <> 'NAO INFORMADO'
    GROUP BY m.MACRORREGIAO, e.NOME_ESPECIALIDADE
),
ranking AS (
    SELECT b.*,
           ROW_NUMBER() OVER (PARTITION BY MACRORREGIAO ORDER BY QTD DESC) AS RK,
           ROUND(100 * RATIO_TO_REPORT(QTD) OVER (PARTITION BY MACRORREGIAO), 2) AS PCT
    FROM base b
)
SELECT MACRORREGIAO, RK, NOME_ESPECIALIDADE, QTD, PCT
FROM ranking
WHERE RK <= 5
"""

In [ ]:
recarregar("MART_P02_TOP_ESPECIALIDADES", SQL, origem="FATO_INTERNACAO")

In [ ]:
consultar("SELECT * FROM MART_P02_TOP_ESPECIALIDADES FETCH FIRST 10 ROWS ONLY")